In [0]:
from pyspark.sql import functions as F

# ── config ──────────────────────────────────────────
CHECKPOINT = "abfss://silver@saretailsalesdev.dfs.core.windows.net/_checkpoint/silver_sales"

# ── read Bronze by TABLE NAME ────────────────────────
bronze = spark.read.table("adb_retail_dev.bronze.sales")

# ── clean ───────────────────────────────────────────
silver = (
    bronze
    .filter(F.col("price").isNotNull())
    .filter(F.col("quantity") > 0)
    .withColumn("order_date",
        F.expr("try_to_date(order_date, 'yyyy-MM-dd')"))
    .filter(F.col("order_date").isNotNull())
    .withColumn("revenue",
        F.col("quantity") * F.col("price"))
    .dropDuplicates(["order_id", "order_date"])
)

# ── write Silver by TABLE NAME ───────────────────────
silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("adb_retail_dev.silver.sales")

print(f"Silver rows: {silver.count()}")
silver.groupBy("order_date").count().orderBy("order_date").show()